# Model Comparison: Merton, GARCH, EDM Diffusion

Loads pre-generated samples from three model families and compares them
against ground-truth close prices.

- **Merton-X**: jump-diffusion with OHL-driven variance
- **GARCH-X**: log-GARCH with Student-t innovations and OHL conditioning
- **EDM**: denoising diffusion probabilistic model (OHLC-conditional)

All models output `(window_idx, sample_idx, step_000 … step_{L-1})` CSVs.  
GT close for Merton/GARCH is reconstructed from the processed input CSVs  
via the `(file, window_start)` metadata embedded in each generated row.  
For EDM, GT comes from the companion `*_gt_ohlc.csv`.

**Normalized domain**  
2A. Aggregate distribution statistics  
2B. Marginal distribution of close — histogram, QQ, ECDF + KS  
2C. Marginal distribution of increments — same  
2D. FTS metrics — `sf.distribution`, `sf.acf`, `sf.leverage_effect`

**Unnormalized domain**  
3A. Aggregate statistics  
3B. Marginal distribution  
3C. CRPS  
3D. Interval coverage

In [ ]:
import os, sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

import replication.stylized_facts as sf

## Configuration

In [ ]:
# ── Merton ────────────────────────────────────────────────────────────────────
MERTON_CSV        = "../data/generated/merton/generated_close.csv"
MERTON_NORM_STATS = "../data/general/normalization_stats_SNP500_normalized.csv"

# ── GARCH (set GARCH_CSV=None to skip if data not yet generated) ──────────────
GARCH_CSV         = "../data/generated/garch/generated_close.csv"
GARCH_NORM_STATS  = "../data/general/normalization_stats_SNP500_normalized.csv"

# ── EDM (diffusion) ───────────────────────────────────────────────────────────
EDM_FOLDER    = "../data/generated/edm/EDM_REPL__CLOS_ep-99_step-6534_lr-8e-04_ch-128_layers-6_20260514_201539_rev_steps_400/csv500_samples50_seed50_20260517_200645"
EDM_SPLIT     = "val"    # "train" | "val"
EDM_NORM_STATS = "../data/general/normalization_stats_derived.csv"

# ── Processed data folder (GT reconstruction for Merton/GARCH) ───────────────
PROCESSED_FOLDER = "../data/SNP500_individual_processed"

# ── Analysis parameters ───────────────────────────────────────────────────────
MODEL_COLORS    = {"Merton": "tab:blue", "GARCH": "tab:orange", "EDM": "tab:green"}
ACF_MAX_LAG     = 50    # max lag for sf.acf
LEV_MAX_LAG     = 50    # max lag for sf.leverage_effect
MAX_SERIES_FTS  = 200   # paths subsampled for FTS (for speed)
COVERAGE_LEVELS = [0.50, 0.80, 0.90, 0.95]

FTS_DIR = "../images/model_comparison/fts"
os.makedirs(FTS_DIR, exist_ok=True)

## 1. Data Loading

In [ ]:
def load_norm_stats(path, feature="close"):
    """Return (mean, std) for a feature from a normalization_stats CSV."""
    df = pd.read_csv(path, index_col=0)
    return float(df.loc[feature, "mean"]), float(df.loc[feature, "std"])


def load_flat_csv(csv_path):
    """
    Load a Merton/GARCH generated_close.csv.
    Returns gen_3d (n_windows, n_samples, seq_len) and a metadata DataFrame
    with columns [window_idx, file, window_start].
    """
    df = pd.read_csv(csv_path)
    step_cols = [c for c in df.columns if c.startswith("step_")]
    seq_len   = len(step_cols)

    window_ids = sorted(df["window_idx"].unique())
    n_samples  = df["sample_idx"].nunique()
    n_windows  = len(window_ids)

    gen_3d = np.zeros((n_windows, n_samples, seq_len), dtype=np.float32)
    meta   = []
    for wi, wid in enumerate(window_ids):
        sub = df[df["window_idx"] == wid].sort_values("sample_idx")
        gen_3d[wi] = sub[step_cols].to_numpy(dtype=np.float32)
        r = sub.iloc[0]
        meta.append({"window_idx": wid, "file": r["file"],
                     "window_start": int(r["window_start"])})

    print(f"  windows={n_windows}, samples/window={n_samples}, seq_len={seq_len}")
    return gen_3d, pd.DataFrame(meta)


def reconstruct_gt_close(meta_df, processed_folder, seq_len):
    """
    Reconstruct GT close array for Merton/GARCH by reading processed CSVs
    and slicing each window using its (file, window_start) metadata.
    Returns gt_close shaped (n_windows, seq_len).
    """
    n_windows = len(meta_df)
    gt_close  = np.zeros((n_windows, seq_len), dtype=np.float32)
    cache     = {}
    for wi, row in meta_df.reset_index(drop=True).iterrows():
        fname, ws = row["file"], row["window_start"]
        if fname not in cache:
            tmp = pd.read_csv(os.path.join(processed_folder, fname))
            tmp["date"] = pd.to_datetime(tmp["date"], format="%d/%m/%Y")
            cache[fname] = tmp.sort_values("date").reset_index(drop=True)["close"].values.astype(np.float32)
        gt_close[wi] = cache[fname][ws : ws + seq_len]
    return gt_close


def load_edm(folder_path, split):
    """
    Load EDM generated close and GT close from the companion OHLC CSV.
    Returns gen_3d (n_windows, n_samples, seq_len) and gt_close (n_windows, seq_len).
    """
    gen_df  = pd.read_csv(os.path.join(folder_path, f"{split}_generated_close.csv"))
    ohlc_df = pd.read_csv(os.path.join(folder_path, f"{split}_gt_ohlc.csv"))
    step_cols = [c for c in gen_df.columns if c.startswith("step_")]
    seq_len   = len(step_cols)

    window_ids = sorted(gen_df["window_idx"].unique())
    n_samples  = gen_df["sample_idx"].nunique()
    n_windows  = len(window_ids)

    gen_3d = np.zeros((n_windows, n_samples, seq_len), dtype=np.float32)
    for wi, wid in enumerate(window_ids):
        sub = gen_df[gen_df["window_idx"] == wid].sort_values("sample_idx")
        gen_3d[wi] = sub[step_cols].to_numpy(dtype=np.float32)

    gt_close = (
        ohlc_df[ohlc_df["feature"] == "close"]
        .sort_values("window_idx")[step_cols]
        .to_numpy(dtype=np.float32)
    )
    print(f"  windows={n_windows}, samples/window={n_samples}, seq_len={seq_len}")
    return gen_3d, gt_close

In [ ]:
# models dict: name -> {gen, gt, norm_stats}
models = {}

for name, csv_path, norm_stats in [
    ("Merton", MERTON_CSV, MERTON_NORM_STATS),
    ("GARCH",  GARCH_CSV,  GARCH_NORM_STATS),
]:
    if csv_path is None or not os.path.isfile(csv_path):
        print(f"SKIP {name}: {csv_path!r} not found")
        continue
    print(f"Loading {name}...")
    gen, meta = load_flat_csv(csv_path)
    gt        = reconstruct_gt_close(meta, PROCESSED_FOLDER, gen.shape[2])
    models[name] = {"gen": gen, "gt": gt, "norm_stats": norm_stats}

print("Loading EDM...")
try:
    gen, gt = load_edm(EDM_FOLDER, EDM_SPLIT)
    models["EDM"] = {"gen": gen, "gt": gt, "norm_stats": EDM_NORM_STATS}
except Exception as e:
    print(f"SKIP EDM: {e}")

print(f"\nLoaded models: {list(models.keys())}")

---
## 2. Normalized Analysis

All analyses below operate on the data in each model's own normalized domain.
Models may use different normalization schemes, so cross-model comparisons
here are qualitative; quantitative comparison happens after unnormalization (§3).

In [ ]:
def core_stats(x, label):
    """Return a dict of key distributional statistics for array x."""
    return {
        "label"          : label,
        "mean"           : float(np.mean(x)),
        "std"            : float(np.std(x)),
        "skewness"       : float(scipy_stats.skew(x)),
        "excess_kurtosis": float(scipy_stats.kurtosis(x, fisher=True)),
        "q01"            : float(np.quantile(x, 0.01)),
        "q05"            : float(np.quantile(x, 0.05)),
        "median"         : float(np.median(x)),
        "q95"            : float(np.quantile(x, 0.95)),
        "q99"            : float(np.quantile(x, 0.99)),
    }


def marginal_plots(gen_flat, gt_flat, name, color, axes_row):
    """Fill one (hist | QQ | ECDF) row for a single model."""
    combined = np.concatenate([gen_flat, gt_flat])
    lo, hi   = np.quantile(combined, [0.001, 0.999])
    bins     = np.linspace(lo, hi, 60)
    probs    = np.linspace(0.01, 0.99, min(5_000, len(gen_flat)))
    ks, ks_p = scipy_stats.ks_2samp(gen_flat, gt_flat)

    # Histogram
    axes_row[0].hist(gt_flat,  bins=bins, density=True, alpha=0.5, color="grey",  label="GT")
    axes_row[0].hist(gen_flat, bins=bins, density=True, alpha=0.5, color=color, label=name)
    axes_row[0].set_title(f"{name} — histogram")
    axes_row[0].legend(fontsize=8)

    # QQ
    q_gt  = np.quantile(gt_flat,  probs)
    q_gen = np.quantile(gen_flat, probs)
    lims  = [min(q_gt.min(), q_gen.min()), max(q_gt.max(), q_gen.max())]
    axes_row[1].scatter(q_gt, q_gen, s=3, alpha=0.4, color=color)
    axes_row[1].plot(lims, lims, "r--", lw=1.2, label="y = x")
    axes_row[1].set_xlabel("GT quantiles"); axes_row[1].set_ylabel("Gen quantiles")
    axes_row[1].set_title(f"{name} — QQ")
    axes_row[1].legend(fontsize=8)

    # ECDF
    for vals, lbl, c in [(gt_flat, "GT", "grey"), (gen_flat, name, color)]:
        s = np.sort(vals)
        axes_row[2].plot(s, np.arange(1, len(s) + 1) / len(s), lw=1.2, color=c, label=lbl)
    axes_row[2].set_xlim(lo, hi)
    axes_row[2].set_title(f"{name} — ECDF  (KS={ks:.3f}, p={ks_p:.2e})")
    axes_row[2].legend(fontsize=8)

### 2A. Aggregate Distribution Statistics

Core moments pooled over all windows × samples × steps (generated)
and all windows × steps (GT). Each model is in its own normalized domain.

In [ ]:
rows = []
for name, m in models.items():
    rows.append(core_stats(m["gen"].ravel(), f"{name} — generated"))
    rows.append(core_stats(m["gt"].ravel(),  f"{name} — GT"))

display(pd.DataFrame(rows).set_index("label").round(5))

### 2B. Marginal Distribution of Close Values

One row per model: histogram (gen vs GT), QQ plot, and ECDF with KS statistic.

In [ ]:
n = len(models)
fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
if n == 1: axes = axes[np.newaxis, :]

for ri, (name, m) in enumerate(models.items()):
    marginal_plots(m["gen"].ravel(), m["gt"].ravel(),
                   name, MODEL_COLORS[name], axes[ri])

fig.suptitle("Marginal distribution — Close values (normalized)", y=1.01)
plt.tight_layout()
display(fig); plt.close(fig)

### 2C. Marginal Distribution of Increments

First differences `\u0394_t = x_{t+1} - x_t` along the sequence axis capture
one-step dynamics. Comparing generated vs GT tests whether the model reproduces
the short-horizon return distribution.

In [ ]:
n = len(models)
fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
if n == 1: axes = axes[np.newaxis, :]

for ri, (name, m) in enumerate(models.items()):
    gen_inc = np.diff(m["gen"], axis=-1).ravel()
    gt_inc  = np.diff(m["gt"],  axis=-1).ravel()
    marginal_plots(gen_inc, gt_inc, name, MODEL_COLORS[name], axes[ri])

fig.suptitle("Marginal distribution — Increments \u0394close (normalized)", y=1.01)
plt.tight_layout()
display(fig); plt.close(fig)

### 2D. FTS Metrics

Three canonical stylized facts using `replication.stylized_facts`:

- **`sf.distribution`**: fat-tail PDF on log scale (internally z-scored — scale-invariant)
- **`sf.acf`**: ACF of |values| averaged across paths (volatility clustering)
- **`sf.leverage_effect`**: cross-correlation L(\u03c4) = Cov(r_t, r_{t+\u03c4}\u00b2) / Var(r\u00b2)

Each function saves an individual plot per model/source and returns data used
for the overlay comparison figures below.

In [ ]:
rng = np.random.default_rng(0)

def make_path_obj(arr_2d, max_series=None):
    """Convert (N, L) array to object array of 1-D paths, optionally subsampling."""
    if max_series is not None and len(arr_2d) > max_series:
        idx = rng.choice(len(arr_2d), max_series, replace=False)
        arr_2d = arr_2d[idx]
    obj = np.empty(len(arr_2d), dtype=object)
    for i, p in enumerate(arr_2d):
        obj[i] = p
    return obj

In [ ]:
# ── sf.distribution (fat-tail PDF) ────────────────────────────────────────────
# One call per model×source saves an individual log-scale PDF plot.
# The returned (dist_x, dist_y) are used to build the comparison overlay.

dist_res = {}
for name, m in models.items():
    nw, ns, sl = m["gen"].shape
    for source, arr in [("gen", m["gen"].ravel()), ("gt", m["gt"].ravel())]:
        dist_res[f"{name}_{source}"] = sf.distribution(
            arr,
            file_name=os.path.join(FTS_DIR, f"distribution_{name}_{source}"),
            scale="log", multiple=False, normalize=True, granuality=100,
        )

# Comparison overlay (positive + negative tails)
for sign, mask_fn, suffix in [(1, lambda x: x > 0, "pos"), (-1, lambda x: x < 0, "neg")]:
    fig, ax = plt.subplots(figsize=(8, 5))
    for name in models:
        color = MODEL_COLORS[name]
        for source, ls, lw in [("gt", "--", 1.2), ("gen", "-", 1.8)]:
            x, y = dist_res[f"{name}_{source}"]
            mask = mask_fn(x)
            label = f"{name}" if source == "gen" else None
            ax.plot(sign * x[mask], y[mask], color=color, ls=ls, lw=lw, label=label)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"Normalized scale $\sigma$"); ax.set_ylabel("PDF")
    ax.set_title(f"Fat-tail PDF — {'positive' if sign > 0 else 'negative'} tail\n"
                 "(solid=generated, dashed=GT)")
    ax.legend(fontsize=9)
    display(fig); plt.close(fig)

In [ ]:
# ── sf.acf (volatility clustering: ACF of |values|) ───────────────────────────

acf_res = {}
for name, m in models.items():
    nw, ns, sl = m["gen"].shape
    gen_paths = make_path_obj(m["gen"].reshape(nw * ns, sl), MAX_SERIES_FTS)
    gt_paths  = make_path_obj(m["gt"], MAX_SERIES_FTS)
    for source, paths in [("gen", gen_paths), ("gt", gt_paths)]:
        acf_res[f"{name}_{source}"] = sf.acf(
            paths,
            file_name=os.path.join(FTS_DIR, f"acf_{name}_{source}"),
            for_abs=True, multiple=True, fit=False, scale="log", max_lag=ACF_MAX_LAG,
        )

lags = np.arange(1, ACF_MAX_LAG + 1)
fig, ax = plt.subplots(figsize=(9, 5))
for name in models:
    color = MODEL_COLORS[name]
    ax.plot(lags, acf_res[f"{name}_gt"],  color=color, ls="--", lw=1.2)
    ax.plot(lags, acf_res[f"{name}_gen"], color=color, ls="-",  lw=1.8, label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Lag $k$"); ax.set_ylabel("ACF of |values|")
ax.set_title("Volatility clustering — solid=generated, dashed=GT")
ax.legend(fontsize=9)
display(fig); plt.close(fig)

In [ ]:
# ── sf.leverage_effect ────────────────────────────────────────────────────────

lev_res = {}
for name, m in models.items():
    nw, ns, sl = m["gen"].shape
    gen_paths = make_path_obj(m["gen"].reshape(nw * ns, sl), MAX_SERIES_FTS)
    gt_paths  = make_path_obj(m["gt"], MAX_SERIES_FTS)
    for source, paths in [("gen", gen_paths), ("gt", gt_paths)]:
        lev_res[f"{name}_{source}"] = sf.leverage_effect(
            paths,
            file_name=os.path.join(FTS_DIR, f"leverage_{name}_{source}"),
            multiple=True, min_lag=1, max_lag=LEV_MAX_LAG,
        )

lag_axis = np.arange(1, LEV_MAX_LAG)
fig, ax = plt.subplots(figsize=(9, 5))
for name in models:
    color = MODEL_COLORS[name]
    ax.plot(lag_axis, lev_res[f"{name}_gt"],  color=color, ls="--", lw=1.2)
    ax.plot(lag_axis, lev_res[f"{name}_gen"], color=color, ls="-",  lw=1.8, label=name)
ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("Lag $\\tau$"); ax.set_ylabel("$L(\\tau)$")
ax.set_title("Leverage effect — solid=generated, dashed=GT")
ax.legend(fontsize=9)
display(fig); plt.close(fig)

---
## 3. Unnormalized Analysis

Apply the inverse z-score: `x_real = x_norm \u00d7 std + mean` using the per-model
normalization stats file.  After this step all values are in raw log-return space
and cross-model comparisons are quantitatively valid.

In [ ]:
for name, m in models.items():
    mu, sigma    = load_norm_stats(m["norm_stats"], feature="close")
    m["gen_raw"] = m["gen"] * sigma + mu
    m["gt_raw"]  = m["gt"]  * sigma + mu
    print(f"{name}: \u03bc={mu:.6f}, \u03c3={sigma:.6f}  "
          f"gen range=[{m['gen_raw'].min():.4f}, {m['gen_raw'].max():.4f}]")

### 3A. Aggregate Statistics (Unnormalized)

In [ ]:
rows_raw = []
for name, m in models.items():
    rows_raw.append(core_stats(m["gen_raw"].ravel(), f"{name} — generated"))
    rows_raw.append(core_stats(m["gt_raw"].ravel(),  f"{name} — GT"))

display(pd.DataFrame(rows_raw).set_index("label").round(6))

### 3B. Marginal Distribution (Unnormalized)

Same three-panel analysis as §2B but in raw log-return space, enabling
direct cross-model comparison.

In [ ]:
n = len(models)
fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
if n == 1: axes = axes[np.newaxis, :]

for ri, (name, m) in enumerate(models.items()):
    marginal_plots(m["gen_raw"].ravel(), m["gt_raw"].ravel(),
                   name, MODEL_COLORS[name], axes[ri])

fig.suptitle("Marginal distribution — Close values (unnormalized log-returns)", y=1.01)
plt.tight_layout()
display(fig); plt.close(fig)

### 3C. CRPS — Probabilistic Forecast Quality

The Continuous Ranked Probability Score measures the quality of the full
predictive distribution against the observation:

$$\text{CRPS}(\{x_i\}, y) = \mathbb{E}[|X - y|] - \tfrac{1}{2}\,\mathbb{E}[|X - X'|]$$

- **Lower is better**; a perfect point forecast collapses to MAE.
- `Term1 = E[|X \u2212 y|]` penalises inaccuracy; `\u00bd Term2 = \u00bd E[|X \u2212 X'|]` rewards spread.
- Computed in unnormalized space so scores are cross-model comparable.

In [ ]:
def crps_ensemble_batch(samples, obs):
    """
    CRPS at every (window, step) position.

    Parameters
    ----------
    samples : (n_windows, n_samples, seq_len)
    obs     : (n_windows, seq_len)

    Returns
    -------
    crps  : (n_windows, seq_len)
    term1 : (n_windows, seq_len)  E[|X - y|]
    term2 : (n_windows, seq_len)  E[|X - X'|]
    """
    term1 = np.mean(np.abs(samples - obs[:, np.newaxis, :]), axis=1)
    term2 = np.zeros_like(term1)
    for w in range(samples.shape[0]):
        s = samples[w]
        diff = np.abs(s[:, np.newaxis, :] - s[np.newaxis, :, :])
        term2[w] = diff.mean(axis=(0, 1))
    return term1 - 0.5 * term2, term1, term2


steps = np.arange(models[next(iter(models))]["gen"].shape[2])

crps_summary   = []
crps_per_step  = {}

for name, m in models.items():
    crps_grid, t1, t2 = crps_ensemble_batch(m["gen_raw"], m["gt_raw"])
    mean_crps = float(crps_grid.mean())
    mean_mae  = float(np.abs(m["gen_raw"].mean(axis=1) - m["gt_raw"]).mean())
    crps_summary.append({
        "model"             : name,
        "CRPS"              : round(mean_crps, 6),
        "MAE (mean fcst)"   : round(mean_mae, 6),
        "spread skill gap"  : round(mean_mae - mean_crps, 6),
        "mean Term1 E[|X-y|]": round(float(t1.mean()), 6),
        "mean Term2 E[|X-X'|]": round(float(t2.mean()), 6),
    })
    crps_per_step[name] = crps_grid.mean(axis=0)

print("CRPS summary (unnormalized space — lower CRPS is better)")
display(pd.DataFrame(crps_summary).set_index("model"))

fig, ax = plt.subplots(figsize=(10, 4))
for name, curve in crps_per_step.items():
    ax.plot(steps, curve, color=MODEL_COLORS[name], lw=1.4, label=name)
ax.set_xlabel("step t"); ax.set_ylabel("mean CRPS")
ax.set_title("CRPS per step (lower = better)")
ax.legend(fontsize=9); ax.grid(True, linewidth=0.3)
display(fig); plt.close(fig)

### 3D. Interval Coverage — Calibration Check

At each nominal level $1-\alpha$ the empirical interval is
$[q_{\alpha/2},\, q_{1-\alpha/2}]$ of the ensemble, computed per (window, step).

- **Empirical \u2248 nominal** \u2192 well-calibrated.
- **Empirical < nominal** \u2192 overconfident (intervals too narrow).
- **Empirical > nominal** \u2192 underconfident (intervals too wide).

In [ ]:
def empirical_coverage(samples, gt, levels):
    """Return DataFrame of (nominal, empirical, gap, mean_width) for each level."""
    rows = []
    for level in levels:
        alpha = 1 - level
        lo = np.quantile(samples, alpha / 2,     axis=1)   # (n_w, L)
        hi = np.quantile(samples, 1 - alpha / 2, axis=1)
        emp   = float(((gt >= lo) & (gt <= hi)).mean())
        width = float((hi - lo).mean())
        rows.append({"nominal": level, "empirical": round(emp, 4),
                     "gap": round(emp - level, 4), "mean_width": round(width, 6)})
    return pd.DataFrame(rows)


cov_dfs = {}
for name, m in models.items():
    cov_dfs[name] = empirical_coverage(m["gen_raw"], m["gt_raw"], COVERAGE_LEVELS)

# Summary table: empirical coverage per model
summary_emp = pd.concat(
    {name: df.set_index("nominal")["empirical"] for name, df in cov_dfs.items()},
    axis=1
)
summary_emp.index.name = "nominal"
print("Empirical coverage (gap > 0 = underconfident | gap < 0 = overconfident)")
display(summary_emp.round(4))

# Calibration curves
n = len(models)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=True)
if n == 1: axes = [axes]
for ax, (name, df) in zip(axes, cov_dfs.items()):
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="perfect")
    ax.plot(df["nominal"], df["empirical"], "o-",
            color=MODEL_COLORS[name], lw=1.6, ms=7, label=name)
    ax.set_xlabel("nominal coverage")
    ax.set_ylabel("empirical coverage")
    ax.set_title(f"{name}")
    ax.legend(fontsize=9); ax.grid(True, linewidth=0.3)
fig.suptitle("Interval coverage calibration (unnormalized space)", y=1.02)
plt.tight_layout()
display(fig); plt.close(fig)